# Guven 100-Hz time-cross decoding × TW correlation

本 notebook 检验 **0.75–1.85 s** 同窗空间内，Guven alpha TW 与 pooled-session single-trial time-cross decoding 的三维相关。TW、training time 和 test time 三条轴均限制为 0.75–1.85 s。

分别运行两套探索性分析：①保留 `Training time = Test time` 主对角线；②在 cluster sign-flip 前将主对角线排除。每个 session 内先计算 matched-trial Pearson correlation，再做 Fisher-z，最后在被试内等权平均四个 session。

In [ ]:
from pathlib import Path
import gc
import importlib
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t as student_t
from IPython.display import display

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/guven/corr/cross_corr'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import guven_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)

TW_DIR = PROJECT_DIR / 'results/guven_fft'
DECODING_DIR = PROJECT_DIR / 'results/guven_alpha_time_cross_fir_timepoint_100hz_toi0_245_pooled_sessions_trialwise_only'
OUTPUT_DIR = DECODING_DIR / 'tw_cross_correlation_tw0p75-1p85_dec0p75-1p85'
FIGURE_DIR = OUTPUT_DIR / 'fig'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

WINDOW = (0.0, 2.45)
FMIN, FMAX = 8.0, 12.0
TW_STRIDE = DECODING_STRIDE = 1
N_PERMUTATIONS = 1000
CLUSTER_ALPHA = CLUSTER_P = 0.05
SEED = 42
POOL_RAW_TRIALS_ACROSS_SESSIONS = False
MEASURES = ('fw', 'bw')
COMPONENTS = ('difference', 'contra', 'ipsi', 'midline', 'all_lines')
CONDITIONS = ('cue', 'uncue', 'cue_minus_uncue')
ANALYSES = tuple(
    (measure, component, condition)
    for measure in MEASURES
    for component in COMPONENTS
    for condition in CONDITIONS
)

decoding_files = sorted(DECODING_DIR.glob('subject_*_alpha_time_cross_torch.npz'))
if len(decoding_files) < 2:
    raise FileNotFoundError(
        f'Need at least two decoding files, found {len(decoding_files)} in {DECODING_DIR}'
    )
with np.load(decoding_files[0], allow_pickle=True) as first:
    available_time = np.asarray(first['time_dec'], dtype=float)
if available_time[0] > WINDOW[0] + 1e-6 or available_time[-1] < WINDOW[1] - 0.0101:
    raise ValueError(
        f'Decoding does not cover requested window {WINDOW}; available ' 
        f'{available_time[0]:.3f}–{available_time[-1]:.3f} s'
    )
print(f'Decoding subjects: {len(decoding_files)}')
print(f'Decoding time: {available_time[0]:.3f}–{available_time[-1]:.3f} s; n={available_time.size}')
print(f'TW/train/test requested window: {WINDOW[0]:.2f}–{WINDOW[1]:.2f} s')
print(f'Combinations in each analysis: {len(ANALYSES)}')

## 共用运行函数

每个组合计算四个 session 内的 matched-trial Pearson correlation，进行 Fisher-z 后得到每位被试的一张三维图，再进行组水平 3D cluster sign-flip。终端会打印显著 cluster 在 TW/training/test 三条轴上的范围，并保存 `3D statistic map + Significant 3D cluster` 双面板 PNG。

In [ ]:
def run_exploratory_3d(*, exclude_diagonal, label, seed_offset):
    results = {}
    for analysis_index, (measure, component, condition) in enumerate(ANALYSES):
        mode = 'OFF-DIAGONAL' if exclude_diagonal else 'WITH-DIAGONAL'
        print(
            f'\n=== {mode}: {measure.upper()} / {component} / {condition} ===',
            flush=True,
        )
        subjects, tw_time, train_time, test_time, subject_z, matched_counts = (
            cross_corr.build_subject_cross_maps(
                TW_DIR, DECODING_DIR, measure=measure, component=component,
                condition=condition, fmin=FMIN, fmax=FMAX,
                tw_limits=WINDOW, train_limits=WINDOW, test_limits=WINDOW,
                tw_stride=TW_STRIDE, decoding_stride=DECODING_STRIDE,
                pool_raw_trials_across_sessions=POOL_RAW_TRIALS_ACROSS_SESSIONS,
            )
        )
        off_diagonal_cells = None
        if exclude_diagonal:
            separation = np.abs(train_time[:, None] - test_time[None, :])
            tolerance = 0.25 * min(
                np.median(np.diff(train_time)), np.median(np.diff(test_time))
            )
            off_diagonal = separation > tolerance
            subject_z = np.where(
                off_diagonal[None, None, :, :], subject_z, 0.0
            ).astype(np.float32, copy=False)
            off_diagonal_cells = int(off_diagonal.sum())
            print(
                f'  retained {off_diagonal_cells}/{off_diagonal.size} train×test cells',
                flush=True,
            )
        mean_z, observed_t, clusters, null_max = cross_corr.cluster_signflip_3d(
            subject_z, permutations=N_PERMUTATIONS,
            cluster_alpha=CLUSTER_ALPHA, cluster_p=CLUSTER_P,
            seed=SEED + seed_offset + analysis_index, return_all=True,
        )
        threshold = student_t.ppf(1 - CLUSTER_ALPHA / 2, len(subjects) - 1)
        significant = [cluster for cluster in clusters if cluster['significant']]
        cluster_p_values = sorted(cluster['p'] for cluster in clusters)
        cross_corr.print_significant_cluster_ranges(
            observed_t, clusters, tw_time, train_time, test_time
        )
        tag = f'{measure}_{component}_{condition}'
        figure_path = FIGURE_DIR / f'{label}_{tag}_3d-statistic-cluster.png'
        figure = cross_corr.plot_3d_statistic_cluster_panels(
            observed_t, clusters, tw_time, train_time, test_time,
            threshold, figure_path,
            title=(
                f'{measure.upper()} TW ({component}) × {condition} decoding '
                f'[0.75–1.85 s; {mode}]'
            ),
        )
        result = {
            'shape': observed_t.shape,
            'n_subjects': len(subjects),
            'n_significant': len(significant),
            'minimum_corrected_p': cluster_p_values[0] if cluster_p_values else None,
            'off_diagonal_cells': off_diagonal_cells,
            'figure': figure_path,
        }
        results[(measure, component, condition)] = result
        print(result, flush=True)
        print(f'Figure saved to: {figure_path}', flush=True)
        display(figure)
        plt.close(figure)
        del subject_z, mean_z, observed_t, clusters, null_max, figure
        gc.collect()
    print(f'\n{label} summary ({len(results)} combinations):')
    for key, result in results.items():
        print(key, result)
    return results

## 1. 探索性 with-diagonal 3D correlation（全部 TW components）

在 0.75–1.85 s 同窗空间内检验 FW/BW × `difference`、`contra`、`ipsi`、`midline`、`all_lines` × cue、uncue、cue−uncue 的全部组合，保留 `Training time = Test time` 主对角线。由于同时检验多个方向、component 和 decoding condition，结果应作为探索性分析报告。

In [ ]:
with_diagonal_results = run_exploratory_3d(
    exclude_diagonal=False,
    label='exploratory_tw0p75-1p85_dec0p75-1p85_with-diagonal',
    seed_offset=0,
)

## 2. 探索性 off-diagonal-only 3D correlation（全部 TW components）

分析组合与上面相同，但在三维 cluster sign-flip 前排除 `Training time = Test time` 主对角线。对角线固定为 0，不能形成显著 voxel，也不能连接对角线两侧的 cluster。

In [ ]:
off_diagonal_results = run_exploratory_3d(
    exclude_diagonal=True,
    label='exploratory_tw0p75-1p85_dec0p75-1p85_off-diagonal',
    seed_offset=10000,
)

## 3. FW/BW difference 随时间变化（0.75–1.85 s，可单独运行）

使用与上面三维相关完全相同的 Guven alpha TW 定义：`difference = contra − ipsi`，其中 contra/ipsi 根据每个 trial 的 `cue_loc` 确定。细线表示每位被试，粗线与阴影表示组均值 ± SEM。四个 session 先在被试内按有效 trial 数加权平均。

In [ ]:
# Standalone: safe to run immediately after restarting the kernel.
from pathlib import Path
import importlib
import pickle
import sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT_DIR = Path('/home/dilay/project2/tw')
MODULE_DIR = PROJECT_DIR / 'travelling_waves/tw/fft/guven/corr/cross_corr'
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import guven_tw_cross_correlation as cross_corr
cross_corr = importlib.reload(cross_corr)

TW_DIR = PROJECT_DIR / 'results/guven_fft'
DECODING_DIR = PROJECT_DIR / 'results/guven_alpha_time_cross_fir_timepoint_100hz_toi0p75_1p85_pooled_sessions_trialwise_only'
FIGURE_DIR = DECODING_DIR / 'fig'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
WINDOW = (0.75, 1.85)
FMIN, FMAX = 8.0, 12.0
SESSIONS = (1, 2, 3, 4)
SUBJECTS = range(1, 31)

subject_curves = {'fw': [], 'bw': []}
selected_time = None
for subject in SUBJECTS:
    session_curves = {'fw': [], 'bw': []}
    session_weights = []
    for session in SESSIONS:
        path = TW_DIR / f'sub{subject:02d}_session{session}.pkl'
        with path.open('rb') as stream:
            data = pickle.load(stream)
        time = cross_corr._tw_time(data)
        step = np.median(np.diff(time))
        tolerance = max(abs(step) * 1e-6, np.finfo(float).eps * 100)
        time_mask = (time >= WINDOW[0]-tolerance) & (time <= WINDOW[1]+tolerance)
        if selected_time is None:
            selected_time = time[time_mask]
        elif not np.allclose(selected_time, time[time_mask]):
            raise ValueError(f'TW time mismatch in {path.name}')
        frequency = np.asarray(data['ff'], dtype=float)
        frequency_mask = (frequency >= FMIN) & (frequency <= FMAX)
        keep = np.ones(np.asarray(data['cue_loc']).size, dtype=bool)
        if 'is_bad_epoch' in data:
            keep &= ~np.asarray(data['is_bad_epoch'], dtype=bool)
        if keep.sum() == 0:
            raise ValueError(f'No valid trials in {path.name}')
        for measure in ('fw', 'bw'):
            values = cross_corr._tw_measure(data, measure)
            difference = cross_corr._component_array(
                values, np.asarray(data['cue_loc']), 'difference'
            )
            curve = difference[frequency_mask][:, time_mask, :][:, :, keep].mean(axis=(0, 2))
            session_curves[measure].append(curve)
        session_weights.append(int(keep.sum()))
        del data, values, difference
    for measure in ('fw', 'bw'):
        subject_curves[measure].append(
            np.average(session_curves[measure], axis=0, weights=session_weights)
        )
    print(f'subject {subject:02d}: valid trials={sum(session_weights)}', flush=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharex=True, sharey=True, constrained_layout=True)
styles = {'fw': ('#2166AC', 'FW difference'), 'bw': ('#B2182B', 'BW difference')}
for axis, measure in zip(axes, ('fw', 'bw')):
    curves = np.asarray(subject_curves[measure])
    group_mean = curves.mean(axis=0)
    sem = curves.std(axis=0, ddof=1) / np.sqrt(curves.shape[0])
    color, title = styles[measure]
    for curve in curves:
        axis.plot(selected_time, curve, color=color, lw=.75, alpha=.20)
    axis.fill_between(
        selected_time, group_mean-sem, group_mean+sem,
        color=color, alpha=.22, linewidth=0,
    )
    axis.plot(selected_time, group_mean, color=color, lw=2.8, label='Group mean ± SEM')
    axis.axhline(0, color='.3', lw=1, ls='--')
    axis.set(title=f'{title}: contra − ipsi', xlabel='TW time (s)', xlim=WINDOW)
    axis.spines[['top', 'right']].set_visible(False)
    axis.legend(frameon=False)
axes[0].set_ylabel('Alpha TW strength difference (dB)')
fig.suptitle('Guven alpha TW lateralization, 0.75–1.85 s', fontsize=14)
figure_path = FIGURE_DIR / 'guven_fw_bw_difference_timecourses_tw0p75-1p85.png'
fig.savefig(figure_path, dpi=250, bbox_inches='tight', facecolor='white')
print(f'Saved: {figure_path}', flush=True)
plt.show()